# 🏷️ Decision Trees — Solutions Notebook

**Complete, verified solutions.**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print('Setup complete! ✅')

### Entropy & Gini Functions Solution

In [ ]:
# ✅ SOLUTION: Entropy & Gini
def entropy(y):
    hist = np.bincount(y)
    ps = hist / len(y)
    return -np.sum([p * np.log2(p) for p in ps if p > 0])

def gini(y):
    hist = np.bincount(y)
    ps = hist / len(y)
    return 1.0 - np.sum(ps ** 2)

y_sample = np.array([0, 0, 0, 1, 1, 1, 1, 1])
print(f'Entropy of sample: {entropy(y_sample):.4f}')
print(f'Gini Impurity of sample: {gini(y_sample):.4f}')

### Decision Tree from Scratch Node & Model Solution

In [ ]:
# ✅ SOLUTION: Simple Node-based Decision Tree Classifier
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        
    def is_leaf_node(self):
        return self.value is not None

class DecisionTreeFromScratch:
    def __init__(self, min_samples_split=2, max_depth=100):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.root = None
        
    def fit(self, X, y):
        self.root = self._grow_tree(X, y)
        
    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))
        
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_val = self._most_common_label(y)
            return Node(value=leaf_val)
            
        feat_idxs = np.random.choice(n_feats, n_feats, replace=False)
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)
        
        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)
        
    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_thresh = 0, 0
        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            for thresh in thresholds:
                gain = self._information_gain(y, X_column, thresh)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thresh
        return split_idx, split_thresh
        
    def _information_gain(self, y, X_column, threshold):
        parent_entropy = entropy(y)
        left_idxs, right_idxs = self._split(X_column, threshold)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = entropy(y[left_idxs]), entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r
        return parent_entropy - child_entropy
        
    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs
        
    def _most_common_label(self, y):
        return np.bincount(y).argmax()
        
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
        
    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

# Test Tree on Iris dataset
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf_custom = DecisionTreeFromScratch(max_depth=5)
clf_custom.fit(X_train, y_train)
predictions = clf_custom.predict(X_test)

print(f'Custom Decision Tree Accuracy on Iris: {accuracy_score(y_test, predictions):.4f}')

### Plotting Tree with scikit-learn

In [ ]:
clf_sk = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
clf_sk.fit(X_train, y_train)

plt.figure(figsize=(12, 8))
plot_tree(clf_sk, feature_names=iris.feature_names, class_names=iris.target_names, filled=True)
plt.title('scikit-learn Decision Tree Visualization')
plt.show()